# Lesson 08: ETL Pipeline — Assignment History

**Step 6 del ejercicio**: Extraer tickets y ticket_assignments de FreeSQL,
encontrar el agente responsable en cada punto en el tiempo,
agregar por fecha y cargar en `fact_ticket_daily`.

---

## El problema del punto en el tiempo

Un ticket puede cambiar de agente. Si Alice creó el ticket y Bob lo resolvió,
`tickets.assigned_to` solo muestra al último asignado (Bob).
Para saber quién era el responsable en el momento de **creación** hay que
consultar `ticket_assignments` con la condición:

```
valid_from <= momento_buscado
AND (valid_to IS NULL OR valid_to > momento_buscado)
```

Eso es lo que hace la función `get_agent_at()` de este notebook.

## Paso 0 — Instalar dependencias

In [ ]:
# oracledb es el driver oficial de Oracle para Python (reemplaza cx_Oracle)
!pip install oracledb pandas --quiet

## Paso 0B — Importar librerías

In [ ]:
import oracledb
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas correctamente.')

## Paso 0C — Conexión a FreeSQL

Reemplaza `DB_USER`, `DB_PASSWORD` y `DB_DSN` con los datos de tu cuenta en FreeSQL.

In [ ]:
DB_USER     = 'tu_usuario'        # Reemplazar
DB_PASSWORD = 'tu_contraseña'     # Reemplazar
DB_DSN      = 'host:port/FREEPDB1'  # Reemplazar con el DSN de FreeSQL

conn = oracledb.connect(
    user=DB_USER,
    password=DB_PASSWORD,
    dsn=DB_DSN
)
print('Conexión exitosa.')

---

# PHASE 1 — EXTRACT

Extraemos las tres tablas que necesitamos del sistema OLTP.
`pd.read_sql` ejecuta la query y convierte el resultado en un DataFrame.

In [ ]:
# Extraer tickets
df_tickets = pd.read_sql("""
    SELECT
        ticket_id,
        title,
        status,
        priority,
        assigned_to,
        created_at,
        updated_at,
        resolved_at
    FROM tickets
""", conn)

print(f'Tickets extraídos: {len(df_tickets)}')
df_tickets.head()

In [ ]:
# Extraer historial de asignaciones
df_assignments = pd.read_sql("""
    SELECT
        assignment_id,
        ticket_id,
        assigned_to,
        valid_from,
        valid_to
    FROM ticket_assignments
    ORDER BY ticket_id, valid_from
""", conn)

print(f'Registros de asignación extraídos: {len(df_assignments)}')
df_assignments

In [ ]:
# Extraer la dimensión de agentes (para mapear agent_id → agent_key)
df_dim_agent = pd.read_sql("""
    SELECT agent_key, agent_id
    FROM dim_agent
""", conn)

agent_key_map = dict(zip(df_dim_agent['agent_id'], df_dim_agent['agent_key']))
print(f'Agentes en dim_agent: {len(agent_key_map)}')
print('Mapa agent_id → agent_key:', agent_key_map)

---

# PHASE 2 — TRANSFORM

## 2A — Función de búsqueda punto en el tiempo

Para cada ticket necesitamos saber:
- ¿Quién era el agente asignado en el momento de `created_at`? → crédito de **creación**
- ¿Quién era el agente asignado en el momento de `resolved_at`? → crédito de **resolución**

La regla de búsqueda es: encontrar la fila en `ticket_assignments` donde
`valid_from <= momento` AND (`valid_to IS NULL` OR `valid_to > momento`).

In [ ]:
def get_agent_at(ticket_id, at_time, df_assignments):
    """
    Devuelve el assigned_to (agent_id) para un ticket en un momento dado.
    Si at_time es NaT (sin fecha) devuelve None.
    """
    if pd.isna(at_time):
        return None

    mask = (
        (df_assignments['ticket_id'] == ticket_id) &
        (df_assignments['valid_from'] <= at_time) &
        (
            df_assignments['valid_to'].isna() |
            (df_assignments['valid_to'] > at_time)
        )
    )
    rows = df_assignments[mask]

    if rows.empty:
        return None

    return int(rows.iloc[0]['assigned_to'])

In [ ]:
# Aplicar la función a cada ticket
df_tickets['creator_agent_id'] = df_tickets.apply(
    lambda row: get_agent_at(row['ticket_id'], row['created_at'], df_assignments),
    axis=1
)

df_tickets['resolver_agent_id'] = df_tickets.apply(
    lambda row: get_agent_at(row['ticket_id'], row['resolved_at'], df_assignments),
    axis=1
)

# Verificar: el ticket 5 debe mostrar creator≠resolver (fue reasignado)
df_tickets[['ticket_id', 'title', 'creator_agent_id', 'resolver_agent_id']]

## 2B — Función auxiliar: convertir fecha a date_key (YYYYMMDD)

In [ ]:
def date_to_key(dt):
    """Convierte un datetime/Timestamp a entero YYYYMMDD. Devuelve None si NaT."""
    if pd.isna(dt):
        return None
    if hasattr(dt, 'date'):
        dt = dt.date()
    return int(dt.strftime('%Y%m%d'))

## 2C — Construir dos datasets: creaciones y resoluciones

Cada ticket puede contribuir con **dos filas** al fact:
1. Una fila en la fecha de creación (crédito al `creator_agent_id`)
2. Una fila en la fecha de resolución (crédito al `resolver_agent_id`)

Luego unimos ambos y agrupamos para obtener counts diarios.

In [ ]:
# Registros de creación
df_created = df_tickets[df_tickets['creator_agent_id'].notna()].copy()
df_created['date_key']  = df_created['created_at'].apply(date_to_key)
df_created['agent_id']  = df_created['creator_agent_id'].astype(int)
df_created['metric']    = 'created'
df_created = df_created[['date_key', 'agent_id', 'status', 'priority', 'metric']]

# Registros de resolución (solo tickets resueltos)
df_resolved = df_tickets[
    df_tickets['resolved_at'].notna() &
    df_tickets['resolver_agent_id'].notna()
].copy()
df_resolved['date_key'] = df_resolved['resolved_at'].apply(date_to_key)
df_resolved['agent_id'] = df_resolved['resolver_agent_id'].astype(int)
df_resolved['metric']   = 'resolved'
df_resolved = df_resolved[['date_key', 'agent_id', 'status', 'priority', 'metric']]

# Combinar ambos
df_all = pd.concat([df_created, df_resolved], ignore_index=True)
print(f'Registros a agregar: {len(df_all)}')
df_all.head(10)

In [ ]:
# Agregar: contar por (date_key, agent_id, status, priority, metric)
df_agg = df_all.groupby(
    ['date_key', 'agent_id', 'status', 'priority', 'metric']
).size().reset_index(name='count')

# Pivot: convertir la columna 'metric' en columnas separadas
df_fact = df_agg.pivot_table(
    index=['date_key', 'agent_id', 'status', 'priority'],
    columns='metric',
    values='count',
    fill_value=0
).reset_index()
df_fact.columns.name = None

# Asegurar que existan ambas columnas aunque no haya datos en uno de los lados
if 'created' not in df_fact.columns:
    df_fact['created'] = 0
if 'resolved' not in df_fact.columns:
    df_fact['resolved'] = 0

df_fact = df_fact.rename(columns={
    'created':  'tickets_created',
    'resolved': 'tickets_resolved'
})

# Agregar agent_key usando el mapa extraído de dim_agent
df_fact['agent_key'] = df_fact['agent_id'].map(agent_key_map)
df_fact = df_fact[df_fact['agent_key'].notna()].copy()
df_fact['agent_key'] = df_fact['agent_key'].astype(int)

print(f'Filas a cargar en fact_ticket_daily: {len(df_fact)}')
df_fact[['date_key', 'agent_key', 'status', 'priority',
         'tickets_created', 'tickets_resolved']]

---

# PHASE 3 — LOAD

`cursor.executemany()` inserta todas las filas en un solo round-trip a la base de datos,
que es más eficiente que hacer un `INSERT` individual por cada fila.

In [ ]:
insert_sql = """
    INSERT INTO fact_ticket_daily
        (date_key, agent_key, status, priority, tickets_created, tickets_resolved)
    VALUES (:1, :2, :3, :4, :5, :6)
"""

rows_to_insert = [
    (
        int(row['date_key']),
        int(row['agent_key']),
        row['status'],
        row['priority'],
        int(row.get('tickets_created', 0)),
        int(row.get('tickets_resolved', 0))
    )
    for _, row in df_fact.iterrows()
]

cursor = conn.cursor()
cursor.executemany(insert_sql, rows_to_insert)
conn.commit()
cursor.close()

print(f'Carga completada: {len(rows_to_insert)} filas insertadas en fact_ticket_daily.')

---

# PHASE 4 — VERIFY (Step 7)

Verificamos que el ticket 5 aparezca correctamente:
- `tickets_created` bajo James (agent_id=2) en la fecha 20260405
- `tickets_resolved` bajo Omar (agent_id=4) en la fecha 20260407

Si las dos filas aparecen con agentes diferentes, el pipeline funcionó correctamente.

In [ ]:
df_verify = pd.read_sql("""
    SELECT
        f.date_key,
        a.agent_name,
        a.team,
        f.status,
        f.priority,
        f.tickets_created,
        f.tickets_resolved
    FROM fact_ticket_daily f
    JOIN dim_agent a ON a.agent_key = f.agent_key
    ORDER BY f.date_key, a.agent_name
""", conn)

df_verify

In [ ]:
# Resumen por agente: totales de creación y resolución
df_summary = pd.read_sql("""
    SELECT
        a.agent_name,
        a.team,
        SUM(f.tickets_created)  AS total_created,
        SUM(f.tickets_resolved) AS total_resolved
    FROM fact_ticket_daily f
    JOIN dim_agent a ON a.agent_key = f.agent_key
    GROUP BY a.agent_name, a.team
    ORDER BY total_resolved DESC
""", conn)

df_summary

In [ ]:
# Cerrar conexión
conn.close()
print('Conexión cerrada. Pipeline ETL completado exitosamente.')